In [ ]:
import optimize
import numpy as np
import pandas as pd

conversion_to_annual = {
    "1d": 252,
    "1wk": 52,
    "1mo": 12,
}
annual_risk_free = 0.02

Current portfolio

In [103]:
# My current portfolio
tickers = ["VWCE.DE", "IUSN.DE"] # current portfolio
interval = "1mo"  # "1d", "1wk", or "1mo"
anualization_factor = conversion_to_annual[interval]

risk_free = (1 + annual_risk_free) ** (1 / anualization_factor) - 1

returns, mu, cov = optimize.download_returns(
    tickers,
    interval,
)
print("min date:", returns.index.min().date())
print("max date:", returns.index.max().date())

stats, annual_mu, annual_cov = optimize.annualized_stats(
    returns,
    interval,
)
print("\nAnnualized ETF statistics:")
print(stats.T.round(4))

weights = np.array([0.074, 0.926])  
current_gm = optimize.geometric_mean_portfolio(weights, mu, cov)
current_sharpe = optimize.sharpe_ratio(weights, mu, cov, risk_free)

print(f'\nCurrent {interval} GM:', round(current_gm, 6))
print("Approx. annualized GM:", round((1 + current_gm) ** anualization_factor - 1, 6))
print(f'Current {interval} Sharpe:', round(current_sharpe, 6))
print("Approx. annualized Sharpe:", round(current_sharpe * np.sqrt(anualization_factor), 6))

[*********************100%***********************]  2 of 2 completed

min date: 2018-05-01
max date: 2026-08-01

Annualized ETF statistics:
Ticker             IUSN.DE  VWCE.DE
historical_return   0.0972   0.1273
arithmetic_return   0.1085   0.1295
volatility          0.1748   0.1349

Current 1mo GM: 0.009888
Approx. annualized GM: 0.125327
Current 1mo Sharpe: 0.228185
Approx. annualized Sharpe: 0.790455


Optimization

In [110]:
# Settings for optimization
tickers = ["VWCE.DE", "IUSN.DE", "EUNA.DE", "PCOM.DE", "PPFB.DE"] 
interval = "1mo"  # "1d", "1wk", or "1mo"
anualization_factor = conversion_to_annual[interval]
risk_free = (1 + annual_risk_free) ** (1 / anualization_factor) - 1

In [111]:
# Optimize
returns, mu, cov = optimize.download_returns(
    tickers,
    interval,
)
print("min date:", returns.index.min().date())
print("max date:", returns.index.max().date())

stats, annual_mu, annual_cov = optimize.annualized_stats(
    returns,
    interval,
)
print("\nAnnualized ETF statistics:")
print(stats.T.round(4))

sharpe = optimize.maximize_sharpe_ratio(mu, cov, risk_free)
gm = optimize.maximize_geometric_mean(mu, cov)

weights = pd.DataFrame(
    {
        "ETF": returns.columns,
        "Sharpe_weight": np.round(sharpe["weights"], 6),
        "GM_weight": np.round(gm["weights"], 6),
    }
).set_index("ETF")
print("\nPortfolio weights:")
print(weights.round(4))
print(f"\nOptimal {interval} Sharpe:", round(sharpe["optimal_sharpe"], 6))
print("Approx. annualized Sharpe:", round(sharpe["optimal_sharpe"] * np.sqrt(anualization_factor), 6))
print(f"Optimal {interval} GM:", round(gm["optimal_gm"], 6))
print("Approx. annualized GM:", round((1 + gm["optimal_gm"]) ** anualization_factor - 1, 6))


[*********************100%***********************]  5 of 5 completed

min date: 2017-12-01
max date: 2026-08-01

Annualized ETF statistics:
Ticker             EUNA.DE  IUSN.DE  PCOM.DE  PPFB.DE  VWCE.DE
historical_return  -0.0021   0.0973   0.0789   0.2016   0.1273
arithmetic_return  -0.0013   0.1086   0.0896   0.1956   0.1295
volatility          0.0416   0.1748   0.1667   0.1472   0.1349

Portfolio weights:
         Sharpe_weight  GM_weight
ETF                              
EUNA.DE         0.0000        0.0
IUSN.DE         0.0000        0.0
PCOM.DE         0.0866        0.0
PPFB.DE         0.5305        1.0
VWCE.DE         0.3828        0.0

Optimal 1mo Sharpe: 0.408612
Approx. annualized Sharpe: 1.415472
Optimal 1mo GM: 0.01541
Approx. annualized GM: 0.201428
